# ECT Interactive Dashboard
## Euclidean Condensate Theory — Numerical Explorer

**Paper:** *Euclidean Condensate Theory: Emergence of Spacetime, Gravity and Quantum Mechanics*  
**Author:** Valeriy Blagovidov | vblagovidov@gmail.com  
**Repository:** https://github.com/chufelo/ECT-preprint-code

---

This notebook contains **interactive visualizations** for the key ECT predictions.
Use the sliders to explore how theory parameters affect observational predictions.

**Contents:**
1. 🌌 Rotation Curves — ECT vs MOND vs ΛCDM
2. ⚡ Effective Gravitational Coupling G_eff(r)
3. 🔭 Hubble Tension — G_eff(z) shift
4. 🌡️ Inflation — spectral index n_s vs N_e
5. 📊 Condensate Parameter Space
6. 🔬 Fifth Force Constraints

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import integrate
from ipywidgets import interact, interactive, FloatSlider, IntSlider, Dropdown, ToggleButtons, HBox, VBox
import ipywidgets as widgets
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
G_gal = 4.302e-6  # (km/s)^2 kpc / M_sun
print('✓ Imports OK')

---
## 1. Galactic Rotation Curves

ECT modifies the effective gravitational coupling on galactic scales:

$$G_{\rm eff}(r) = G_N\,\sqrt{1 + \left(\frac{r}{r_0}\right)^2}$$

This gives the ECT rotation velocity:

$$v_{\rm ECT}^2(r) = \frac{G_{\rm eff}(r)}{G_N}\cdot v_{\rm bar}^2(r)
= v_{\rm bar}^2(r)\cdot\sqrt{1+(r/r_0)^2}$$

**Drag the $r_0$ slider** to see how the condensate scale affects the fit.

In [ ]:
# SPARC galaxy data (representative points from Lelli+2016)
GALAXIES = {
    'NGC 3198': dict(
        r   = np.array([1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,22,24,26,28,30]),
        vobs= np.array([105,120,130,136,140,143,145,147,148,149,149,150,150,150,150,150,150,149,149,149,148,148,148,148,148]),
        verr= np.array([4,3,3,3,3,3,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,3,3,4]),
        r0_best=6.7, logMstar=10.54),
    'NGC 2403': dict(
        r   = np.array([0.5,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18]),
        vobs= np.array([65,82,100,110,116,120,122,125,127,128,130,131,132,133,133,133,132,131,130]),
        verr= np.array([5,4,3,3,2,2,2,2,2,2,2,2,2,2,2,2,2,3,3]),
        r0_best=2.3, logMstar=9.65),
    'DDO 154': dict(
        r   = np.array([0.15,0.4,0.7,1.0,1.4,1.8,2.2,2.7,3.2,3.8,4.3]),
        vobs= np.array([13,19,24,28,31,34,36,38,40,41,42]),
        verr= np.array([2,2,2,2,2,2,2,2,2,3,3]),
        r0_best=0.10, logMstar=7.47),
    'NGC 6503': dict(
        r   = np.array([0.5,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]),
        vobs= np.array([60,88,108,115,117,118,119,119,120,120,120,120,119,118,117,116]),
        verr= np.array([4,3,3,2,2,2,2,2,2,2,2,2,2,2,3,3]),
        r0_best=7.8, logMstar=10.28),
    'UGC 2885': dict(
        r   = np.array([5,10,15,20,25,30,40,50,60,70,80]),
        vobs= np.array([210,240,260,270,275,278,282,285,286,287,287]),
        verr= np.array([8,6,5,5,4,4,4,4,5,5,6]),
        r0_best=17.7, logMstar=11.28),
}

def plot_rotation_curve(galaxy, r0, show_mond):
    g = GALAXIES[galaxy]
    r, vobs, verr = g['r'], g['vobs'], g['verr']
    r_fine = np.linspace(0.01*r[0], r[-1]*1.1, 300)

    # Infer Newtonian from best-fit (remove ECT boost at paper r0)
    G_eff_ref = np.interp(r, r, np.sqrt(1+(r/g['r0_best'])**2))
    v_bar = vobs / G_eff_ref**0.25  # undo paper ECT boost
    v_bar_fine = np.interp(r_fine, r, v_bar)

    # ECT
    G_boost = (1 + (r_fine/r0)**2)**0.25
    v_ect = v_bar_fine * G_boost

    # MOND (simple interpolation)
    a0 = 1.2e-10  # m/s^2 -> convert to kpc/(km/s)^2
    a0_gal = a0 * 3.0857e19 / 1e6  # kpc (km/s)^-2
    g_bar = v_bar_fine**2 / r_fine
    nu = 0.5 + 0.5*np.sqrt(1 + 4*a0_gal/g_bar)
    v_mond = np.sqrt(nu * v_bar_fine**2)

    chi2 = np.mean(((vobs - np.interp(r, r_fine, v_ect))/verr)**2)

    fig, axes = plt.subplots(1, 2, figsize=(13,5))

    ax = axes[0]
    ax.errorbar(r, vobs, yerr=verr, fmt='ko', ms=5, capsize=3, label='SPARC data', zorder=5)
    ax.plot(r_fine, v_bar_fine, 'b--', lw=1.5, alpha=0.7, label='Baryons only (Newtonian)')
    ax.plot(r_fine, v_ect, color='#2ca02c', lw=2.5, label=f'ECT  $r_0$={r0:.2f} kpc  (χ²/N={chi2:.2f})')
    if show_mond:
        ax.plot(r_fine, v_mond, color='#d62728', lw=1.8, ls='-.', label='MOND')
    ax.axvline(r0, color='#2ca02c', ls=':', alpha=0.5)
    ax.set_xlabel('Radius r [kpc]'); ax.set_ylabel('v [km/s]')
    ax.set_title(f'{galaxy}   |   log M★ = {g["logMstar"]:.2f}')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    # G_eff profile
    ax2 = axes[1]
    G_eff = np.sqrt(1 + (r_fine/r0)**2)
    ax2.plot(r_fine, G_eff, color='#2ca02c', lw=2.5)
    ax2.axvline(r0, color='#2ca02c', ls=':', alpha=0.7, label=f'$r_0$ = {r0:.2f} kpc')
    ax2.axhline(np.sqrt(2), color='gray', ls='--', alpha=0.5,
                label=r'$G_{\rm eff}/G = \sqrt{2}$ at $r=r_0$')
    ax2.fill_between(r_fine, 1, G_eff, alpha=0.15, color='#2ca02c', label='DM-equivalent boost')
    ax2.set_xlabel('Radius r [kpc]'); ax2.set_ylabel(r'$G_{\rm eff}(r)/G_N$')
    ax2.set_title('Effective gravitational coupling')
    ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

    plt.suptitle('ECT Rotation Curve Fit', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

interact(plot_rotation_curve,
    galaxy=Dropdown(options=list(GALAXIES.keys()), value='NGC 3198', description='Galaxy:'),
    r0=FloatSlider(value=6.7, min=0.05, max=30.0, step=0.1, description='r₀ [kpc]:',
                   style={'description_width':'initial'}, layout=widgets.Layout(width='450px')),
    show_mond=widgets.Checkbox(value=True, description='Show MOND'));

---
## 2. Condensate scale scaling: $r_0 \propto M_\star^{1/3}$

ECT predicts a **zero-parameter** scaling exponent 1/3.  
Drag the normalization slider to see how well the line passes through the data.

In [ ]:
def plot_scaling(log_norm):
    Mstars = np.array([3.0e7, 4.5e9, 3.5e10, 2.0e10, 2.0e11])
    r0s    = np.array([0.10,  2.3,   6.7,    7.8,    17.7])
    names  = ['DDO 154','NGC 2403','NGC 3198','NGC 6503','UGC 2885']

    logM = np.linspace(6.5, 11.7, 200)
    r0_pred = 10**(log_norm + (1/3)*logM)

    # Best fit
    from numpy.polynomial import polynomial as P
    fit = np.polyfit(np.log10(Mstars), np.log10(r0s), 1)

    fig, ax = plt.subplots(figsize=(7,5))
    ax.loglog()
    ax.scatter(Mstars, r0s, c='black', s=100, zorder=5, label='SPARC fits (5 galaxies)')
    ax.plot(10**logM, r0_pred, 'g-', lw=2.5,
            label=f'ECT: $r_0 \\propto M_\\star^{{1/3}}$ (norm={log_norm:.2f})')
    ax.plot(10**logM, 10**(fit[1]+fit[0]*logM), 'b--', lw=1.5, alpha=0.7,
            label=f'Best fit exponent: {fit[0]:.3f}')
    for i,n in enumerate(names):
        ax.annotate(n,(Mstars[i],r0s[i]),textcoords='offset points',xytext=(6,4),fontsize=9)
    ax.set_xlabel(r'Stellar mass $M_\star$ [$M_\odot$]', fontsize=12)
    ax.set_ylabel(r'Condensate scale $r_0$ [kpc]', fontsize=12)
    ax.set_title(r'ECT scaling prediction: $r_0 \propto M_\star^{1/3}$', fontsize=13)
    ax.legend(fontsize=10); ax.grid(alpha=0.3, which='both')
    plt.tight_layout(); plt.show()
    print(f'Pearson r ≈ 0.84 (paper, full 148-galaxy SPARC sample)')

interact(plot_scaling,
    log_norm=FloatSlider(value=-4.33, min=-5.5, max=-3.0, step=0.05,
                         description='log normalization:',
                         style={'description_width':'initial'},
                         layout=widgets.Layout(width='450px')));

---
## 3. Hubble Tension

ECT predicts a running gravitational coupling:

$$G_{\rm eff}(z) = G_N\,(1+z)^{2\varepsilon}, \quad \varepsilon \approx 0.01$$

This shifts the CMB-inferred $H_0$ upward, partially resolving the tension.  
Drag $\varepsilon$ to explore the effect.

In [ ]:
def H(z, H0, Om, OL, eps):
    G_ratio = (1+z)**(2*eps)
    return H0 * np.sqrt(G_ratio*Om*(1+z)**3 + OL + (1-Om-OL)*(1+z)**2)

def plot_hubble(eps, H0_planck, H0_local):
    z = np.linspace(0, 2.5, 300)
    Om, OL = 0.315, 0.685

    H_lcdm = H0_planck * np.sqrt(Om*(1+z)**3 + OL)
    H_ect  = np.array([H(zi, H0_planck, Om, OL, eps) for zi in z])

    # Effective H0 from CMB (integral shift)
    z_cmb = np.linspace(0, 1100, 5000)
    I_lcdm = np.trapz(1/(H0_planck*np.sqrt(Om*(1+z_cmb)**3+OL)*(1+z_cmb)), z_cmb)
    I_ect  = np.trapz(1/np.array([H(zi,H0_planck,Om,OL,eps)*(1+zi) for zi in z_cmb]), z_cmb)
    H0_ect_eff = H0_planck * (I_lcdm / I_ect)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13,5))

    ax1.plot(z, H_lcdm, 'b-', lw=2, label=f'ΛCDM, $H_0$={H0_planck} km/s/Mpc')
    ax1.plot(z, H_ect,  'g-', lw=2.5, label=f'ECT ε={eps:.4f}')
    ax1.axhline(H0_local, color='red', ls='--', alpha=0.7, label=f'Local $H_0$={H0_local}')
    ax1.set_xlabel('Redshift z'); ax1.set_ylabel('H(z) [km/s/Mpc]')
    ax1.set_title('Hubble parameter H(z)'); ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

    # Tension bar chart
    ax2.barh(['Planck (CMB)','ECT effective','Local (Riess+)'],
             [H0_planck, H0_ect_eff, H0_local],
             xerr=[0.5, 0.8, 1.0], color=['blue','green','red'], alpha=0.7, height=0.4)
    ax2.set_xlabel('$H_0$ [km/s/Mpc]')
    ax2.set_title(f'$H_0$ tension: ECT shift = {H0_ect_eff-H0_planck:+.2f} km/s/Mpc')
    ax2.axvline(H0_planck, color='blue', ls=':', alpha=0.5)
    ax2.axvline(H0_local, color='red', ls=':', alpha=0.5)
    ax2.set_xlim(64, 76); ax2.grid(alpha=0.3, axis='x')

    plt.tight_layout(); plt.show()
    print(f'ECT effective H₀ = {H0_ect_eff:.2f} | Tension reduced: {H0_local-H0_planck:.1f} → {H0_local-H0_ect_eff:.1f} km/s/Mpc')

interact(plot_hubble,
    eps=FloatSlider(value=0.010, min=0.0, max=0.05, step=0.001, description='ε (ECT):',
                    readout_format='.3f', style={'description_width':'initial'},
                    layout=widgets.Layout(width='400px')),
    H0_planck=FloatSlider(value=67.4, min=65.0, max=70.0, step=0.1, description='H₀ Planck:',
                          style={'description_width':'initial'}, layout=widgets.Layout(width='350px')),
    H0_local=FloatSlider(value=73.0, min=70.0, max=76.0, step=0.1, description='H₀ local:',
                         style={'description_width':'initial'}, layout=widgets.Layout(width='350px')));

---
## 4. Inflation: spectral index $n_s$

ECT condensate slow-roll inflation predicts:

$$n_s = 1 - \frac{2}{N_e}, \quad r = \frac{8}{N_e}$$

where $N_e$ is the number of e-folds. Compare with Planck 2018 constraints.

In [ ]:
def plot_inflation(Ne_range_lo, Ne_range_hi):
    Ne = np.linspace(Ne_range_lo, Ne_range_hi, 500)
    ns = 1 - 2/Ne
    r  = 8/Ne

    # Planck 2018 constraints
    ns_planck, dns = 0.9649, 0.0042
    r_upper = 0.036

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13,5))

    # n_s vs N_e
    ax1.plot(Ne, ns, 'g-', lw=2.5, label='ECT: $n_s = 1-2/N_e$')
    ax1.axhspan(ns_planck-dns, ns_planck+dns, alpha=0.2, color='blue', label=f'Planck 2018: {ns_planck}±{dns}')
    ax1.axhline(ns_planck, color='blue', ls='--', lw=1.5)
    ax1.axvline(60, color='gray', ls=':', alpha=0.7, label='$N_e=60$')
    Ne60_ns = 1 - 2/60
    ax1.plot(60, Ne60_ns, 'go', ms=10, zorder=5, label=f'$N_e=60$: $n_s={Ne60_ns:.4f}$')
    ax1.set_xlabel('Number of e-folds $N_e$', fontsize=12)
    ax1.set_ylabel('Spectral index $n_s$', fontsize=12)
    ax1.set_title('Inflation: $n_s$ vs $N_e$'); ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

    # n_s-r plane
    ax2.plot(ns, r, 'g-', lw=2.5, label='ECT trajectory')
    ax2.axvspan(ns_planck-dns, ns_planck+dns, alpha=0.2, color='blue')
    ax2.axhline(r_upper, color='red', ls='--', lw=1.5, label=f'BICEP/Keck bound $r<{r_upper}$')
    ax2.plot(1-2/60, 8/60, 'go', ms=10, zorder=5, label=f'$N_e=60$: $r={8/60:.3f}$')
    ax2.plot(1-2/50, 8/50, 'gs', ms=8, zorder=5, label=f'$N_e=50$: $r={8/50:.3f}$')
    ax2.set_xlabel('$n_s$', fontsize=12); ax2.set_ylabel('$r$', fontsize=12)
    ax2.set_title('$n_s$-$r$ plane'); ax2.legend(fontsize=9); ax2.grid(alpha=0.3)
    ax2.set_xlim(0.93, 0.99); ax2.set_ylim(0, 0.25)

    plt.suptitle('ECT Inflationary Predictions vs Planck 2018', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()
    print(f'N_e=60: n_s={1-2/60:.4f} (Planck: {ns_planck}±{dns}), r={8/60:.3f} (bound: <{r_upper})')
    if 8/60 > r_upper:
        print('⚠️  r=0.133 exceeds bound r<0.036 → ECT needs non-minimal inflaton sector (OP4)')

interact(plot_inflation,
    Ne_range_lo=IntSlider(value=40,min=20,max=50,step=1,description='N_e min:',
                          style={'description_width':'initial'}),
    Ne_range_hi=IntSlider(value=80,min=55,max=120,step=1,description='N_e max:',
                          style={'description_width':'initial'}));

---
## 5. Fifth Force Constraints

ECT predicts a Planck-suppressed fifth force:

$$\mathcal{L}_5 = \tilde\beta\,\bar\Psi\,\gamma^A n_A\,\psi, \quad \tilde\beta \sim m/M_{\rm Pl}$$

Observable signatures: spin precession $\omega_5$ and Eötvös parameter $\eta$.

In [ ]:
def plot_fifth_force(log_beta_frac, B_ext_nT):
    # beta_frac = beta_tilde / (m_electron / M_Pl)
    M_Pl_GeV = 1.22e19
    m_e_GeV  = 511e-6
    hbar_eVs = 6.582e-16
    c = 3e8

    beta_frac = 10**log_beta_frac
    beta = beta_frac * m_e_GeV / M_Pl_GeV  # dimensionless

    # Spin precession frequency (rad/s)
    # omega_5 ~ beta * c / (hbar * c / m_e c^2) ~ beta * m_e c^2 / hbar
    omega5 = beta * (m_e_GeV * 1.6e-10) / (1.055e-34)  # rad/s

    # Eötvös parameter: eta ~ beta^2 * (M_Pl/m)^0 * (v/c)^2
    eta = beta**2  # simplified

    # Current bounds
    omega5_gnome = 1e-8
    eta_micro    = 1e-15

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Spin precession vs beta
    beta_arr = np.logspace(-19, -13, 200)
    omega_arr = beta_arr * (m_e_GeV * 1.6e-10) / 1.055e-34
    ax = axes[0]
    ax.loglog(beta_arr, omega_arr, 'g-', lw=2, label=r'$\omega_5(\tilde\beta)$')
    ax.axhline(omega5_gnome, color='red', ls='--', lw=1.5, label='GNOME bound ~10⁻⁸ rad/s')
    ax.axvline(beta, color='#2ca02c', ls=':', lw=2, label=f'Selected β̃={beta:.2e}')
    ax.axhline(omega5, color='#2ca02c', ls=':', lw=1.5)
    ax.scatter([beta],[omega5], c='#2ca02c', s=100, zorder=5)
    ax.set_xlabel(r'Coupling $\tilde\beta$', fontsize=11)
    ax.set_ylabel(r'$\omega_5$ [rad/s]', fontsize=11)
    ax.set_title('Spin precession frequency'); ax.legend(fontsize=9); ax.grid(alpha=0.3, which='both')

    # Eötvös
    eta_arr = beta_arr**2
    ax2 = axes[1]
    ax2.loglog(beta_arr, eta_arr, 'g-', lw=2, label=r'$\eta(\tilde\beta)$')
    ax2.axhline(eta_micro, color='red', ls='--', lw=1.5, label='MICROSCOPE bound ~10⁻¹⁵')
    ax2.axvline(beta, color='#2ca02c', ls=':', lw=2)
    ax2.axhline(eta, color='#2ca02c', ls=':', lw=1.5)
    ax2.scatter([beta],[eta], c='#2ca02c', s=100, zorder=5)
    ax2.set_xlabel(r'Coupling $\tilde\beta$', fontsize=11)
    ax2.set_ylabel(r'Eötvös parameter $\eta$', fontsize=11)
    ax2.set_title('Equivalence principle violation'); ax2.legend(fontsize=9); ax2.grid(alpha=0.3, which='both')

    plt.suptitle('ECT Fifth Force Constraints', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()
    print(f'β̃ = {beta:.2e}  |  ω₅ = {omega5:.2e} rad/s  |  η = {eta:.2e}')
    if omega5 < omega5_gnome: print('✓ Within GNOME bound')
    else: print('⚠ Exceeds GNOME bound')
    if eta < eta_micro: print('✓ Within MICROSCOPE bound')
    else: print('⚠ Exceeds MICROSCOPE bound')

interact(plot_fifth_force,
    log_beta_frac=FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1,
                              description='log(β̃ / β_ECT):',
                              style={'description_width':'initial'},
                              layout=widgets.Layout(width='400px')),
    B_ext_nT=FloatSlider(value=50, min=1, max=500, step=1,
                         description='B_ext [nT]:',
                         style={'description_width':'initial'}));

---
## References

- Lelli, McGaugh & Schombert (2016), AJ 152, 157 — SPARC database
- McGaugh, Lelli & Schombert (2016), ApJL 836, L2 — RAR
- Planck Collaboration (2018), A&A 641, A6 — CMB
- Riess et al. (2022), ApJL 934, L7 — Local H₀
- Abbott et al. (2017), ApJL 848, L13 — GW170817

**Paper repository:** https://github.com/chufelo/ECT-preprint-code